# Reproducing PerLeadCNN Results

This notebook reproduces the reported metrics for **PerLeadCNN** by
re-evaluating the bundled checkpoints on their exact patient-grouped test
splits, using the same code as `src/` (so the notebook and the scripts can
never drift apart).

Requires the ECG dataset (PHI, not shipped — see `DATA.md`). Set
`REPNET_DATA_DIR` to your data directory before running.


In [1]:
import os, sys, json
import numpy as np
# Make the package importable when running from the package root.
sys.path.insert(0, os.path.abspath('.'))
# Point at the ECG dataset (PHI, not shipped — see DATA.md). The dataset lives
# at the repo root, one level above this package. Edit this path if yours is
# elsewhere.
os.environ['REPNET_DATA_DIR'] = os.path.abspath(
    os.path.join('..', 'data', 'seniordesign_upload'))
assert os.path.isdir(os.environ['REPNET_DATA_DIR']), \
    f"data dir not found: {os.environ['REPNET_DATA_DIR']} — edit REPNET_DATA_DIR above"
from src.data import load_dataset
from src.evaluate import evaluate_checkpoint_on_split, RESULTS_DIR
print('data dir:   ', os.path.relpath(os.environ['REPNET_DATA_DIR']))
print('results dir:', os.path.relpath(RESULTS_DIR))


data dir:    ../data/seniordesign_upload
results dir: results/multisplit_dbb6f49


## 1. Load the recorded results


In [2]:
per_split = json.load(open(os.path.join(RESULTS_DIR, 'per_split.json')))
summary   = json.load(open(os.path.join(RESULTS_DIR, 'summary.json')))
aurocs = [s['auroc'] for s in per_split]
best_i   = int(np.argmax(aurocs))
median_i = int(np.argsort(aurocs)[len(aurocs)//2])
print(f"recorded: AUROC {summary['auroc_mean']:.4f} +/- {summary['auroc_std']:.4f}")
print(f"recorded: AUPRC {summary['auprc_mean']:.4f} +/- {summary['auprc_std']:.4f}")
print(f'best split={best_i}, median split={median_i}, params={summary["num_params"]:,}')


recorded: AUROC 0.7085 +/- 0.0493
recorded: AUPRC 0.3423 +/- 0.0751
best split=17, median split=24, params=29,490


## 2. Load the dataset

Reads every recording, preprocesses, and downsamples to 2500 samples @ 250 Hz.


In [3]:
X, y, groups = load_dataset()
print(f'{len(y)} recordings, {int(y.sum())} positive ({y.mean():.1%}), '
      f'{len(np.unique(groups))} patients, shape {X.shape}')


2178 recordings, 335 positive (15.4%), 1383 patients, shape (2178, 12, 2500)


## 3. Verify the BEST checkpoint (split 17)


In [4]:
repro = evaluate_checkpoint_on_split(best_i, os.path.join(RESULTS_DIR, 'best_model.pt'), X, y, groups)
rec = per_split[best_i]
keys = ['auroc','auprc','youden_sens','youden_spec','youden_f1','sens80_sens','sens80_npv']
print(f"{'metric':<14}{'repro':>10}{'recorded':>10}{'match':>8}")
for k in keys:
    ok = abs(repro[k]-rec[k]) < 1e-4
    print(f'{k:<14}{repro[k]:>10.4f}{rec[k]:>10.4f}{("OK" if ok else "FAIL"):>8}')


metric             repro  recorded   match
auroc             0.7793    0.7793      OK
auprc             0.4852    0.4852      OK
youden_sens       0.5429    0.5429      OK
youden_spec       0.8963    0.8963      OK
youden_f1         0.5170    0.5170      OK
sens80_sens       0.8000    0.8000      OK
sens80_npv        0.9372    0.9372      OK


## 4. Verify the MEDIAN checkpoint (split 24)


In [5]:
repro = evaluate_checkpoint_on_split(median_i, os.path.join(RESULTS_DIR, 'median_model.pt'), X, y, groups)
rec = per_split[median_i]
print(f"{'metric':<14}{'repro':>10}{'recorded':>10}{'match':>8}")
for k in keys:
    ok = abs(repro[k]-rec[k]) < 1e-4
    print(f'{k:<14}{repro[k]:>10.4f}{rec[k]:>10.4f}{("OK" if ok else "FAIL"):>8}')


metric             repro  recorded   match
auroc             0.7251    0.7251      OK
auprc             0.2965    0.2965      OK
youden_sens       0.8205    0.8205      OK
youden_spec       0.5466    0.5466      OK
youden_f1         0.4038    0.4038      OK
sens80_sens       0.8077    0.8077      OK
sens80_npv        0.9339    0.9339      OK


## 5. Verify the aggregate (30-split mean/std)


In [6]:
for name, comp, recd in [('AUROC mean', np.mean(aurocs), summary['auroc_mean']),
                         ('AUROC std',  np.std(aurocs),  summary['auroc_std'])]:
    print(f'{name:<12}{comp:>12.6f}{recd:>12.6f}  {"OK" if abs(comp-recd)<1e-6 else "FAIL"}')


AUROC mean      0.708514    0.708514  OK
AUROC std       0.049281    0.049281  OK


---
For a one-shot command-line version of all of the above, run
`python -m src.evaluate`. To regenerate the figures, run `python -m src.analyze`.
